### Homework 3
Damion Huppert

In [1]:
using Pkg
Pkg.add("Plots")
using Plots
Pkg.add("JuMP")
using JuMP
Pkg.add("HiGHS")
using HiGHS
Pkg.add("DataFrames")
using DataFrames
Pkg.add("CSV")
using CSV
Pkg.add("NamedArrays")
using NamedArrays
Pkg.add("LinearAlgebra")
using LinearAlgebra
Pkg.add("XLSX")
using XLSX
Pkg.add("PrettyTables")
using PrettyTables

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.to

##### Question 1-1

Decision Variables:  
$$
C_t, \text{ Money Barry has at year t }
$$
$$
b_{i}, \text{ bonds of type i bought }
$$
$$
s_{i}, \text{ bonds of type i sold }
$$

Objective:  
$$
\max{C_3}
$$

Subject To:  
$$
C_t \geq 0 \quad \forall t \in T
$$
$$
1000 \geq b_{i} \geq 0  \quad \forall i \in I \text{ and } \forall t \in T
$$
$$
1000 \geq s_{i} \geq 0  \quad \forall i \in I \text{ and } \forall t \in T
$$
$$
C_0 = 100000 - \sum_{i=1}^{4} (\text{Ask Price}_i \cdot b_i) + \sum_{i=1}^{4} (\text{Bid Price}_i \cdot s_i)
$$
$$
C_1 = 1.03 \cdot C_0 + \sum_{i=1}^{4} (\text{Payout}_{i,1} \cdot (b_i - s_i)) - 10000
$$
$$
C_2 = 1.03 \cdot C_1 + \sum_{i=1}^{4} (\text{Payout}_{i,2} \cdot (b_i - s_i)) - 11000
$$
$$
C_3 = 1.03 \cdot C_2 + \sum_{i=1}^{4} (\text{Payout}_{i,3} \cdot (b_i - s_i))
$$

##### Question 1-2

In [2]:
using JuMP, HiGHS

# Define sets
T = 0:3                           # Years
I = 1:4                           # Bond types

ask_price = [980, 970, 960, 940]  # Ask prices of bonds
bid_price = [990, 985, 972, 954]  # Bid prices of bonds
payout = [                        # Payouts per year for each bond
    [100, 110, 1100],
    [80, 90, 1120],
    [70, 80, 1090],
    [60, 50, 1110]
]
expenses = [0, 10000, 11000, 0]   # Expenses at each year
total_initial_cash = 100000       # Initial cash
interest_rate = 1.03              # 3% interest per year

model = Model(HiGHS.Optimizer)

@variable(model, C[T] >= 0)          # Cash at year t
@variable(model, 0 <= b[I] <= 1000)  # Bonds bought
@variable(model, 0 <= s[I] <= 1000)  # Bonds sold

@constraint(model, C[0] == total_initial_cash - sum(ask_price[i] * b[i] for i in I) + sum(bid_price[i] * s[i] for i in I))
@constraint(model, C[1] == interest_rate * C[0] + sum(payout[i][1] * (b[i] - s[i]) for i in I) - expenses[2])
@constraint(model, C[2] == interest_rate * C[1] + sum(payout[i][2] * (b[i] - s[i]) for i in I) - expenses[3])
@constraint(model, C[3] == interest_rate * C[2] + sum(payout[i][3] * (b[i] - s[i]) for i in I))

@objective(model, Max, C[3])

# Solve the model
set_silent(model)
optimize!(model)

# Display results
println("Optimal final cash: ", objective_value(model))
println("Bonds bought:")
for i in I
    println("Bond $i: ", value(b[i]))
end

println("Bonds sold:")
for i in I
    println("Bond $i: ", value(s[i]))
end


Optimal final cash: 222261.00425531916
Bonds bought:
Bond 1: 1000.0
Bond 2: 1000.0
Bond 3: 0.0
Bond 4: 80.85106382978724
Bonds sold:
Bond 1: 0.0
Bond 2: 0.0
Bond 3: 1000.0
Bond 4: 1000.0


##### Question 1-3

Decision Variables:  
$$
C_t, \text{ Money Barry has at year t }
$$
$$
b_{i}, \text{ bonds of type i bought }
$$
$$
s_{i}^{\leq500}, \text{ first 500 bonds of type i sold }
$$
$$
s_{i}^{\gt500}, \text{ last 500 bonds of type i sold }
$$

Objective:  
$$
\max{C_3}
$$

Subject To:  
$$
C_t \geq 0 \quad \forall t \in T
$$
$$
1000 \geq b_{i} \geq 0  \quad \forall i \in I \text{ and } \forall t \in T
$$

$$
500 \geq s_{i}^{\leq500} \geq 0  \quad \forall i \in I \text{ and } \forall t \in T
$$
$$
500 \geq s_{i}^{\gt500} \geq 0  \quad \forall i \in I \text{ and } \forall t \in T
$$
$$
s_{i} = s_{i}^{\leq500} + s_{i}^{\gt500}
$$

$$
C_0 = 100000 - \sum_{i=1}^{4} (\text{Ask Price}_i \cdot b_i) + \sum_{i=1}^{4} (\text{Bid Price}^{\leq 500}_i \cdot s_i^{\leq 500}) + \sum_{i=1}^{4} (\text{Bid Price}^{>500}_i \cdot s_i^{>500})
$$
$$
C_1 = 1.03 \cdot C_0 + \sum_{i=1}^{4} (\text{Payout}_{i,1} \cdot (b_i - s_i)) - 10000
$$
$$
C_2 = 1.03 \cdot C_1 + \sum_{i=1}^{4} (\text{Payout}_{i,2} \cdot (b_i - s_i)) - 11000
$$
$$
C_3 = 1.03 \cdot C_2 + \sum_{i=1}^{4} (\text{Payout}_{i,3} \cdot (b_i - s_i))
$$

##### Question 2-1

Decision Variables:
$$
S_t: \text{number of snitches to produce during period t}
$$
$$
I_t: \text{The increase in production rates between periods t − 1 and t}
$$
$$
D_t: \text{The decrease in production rates between periods t − 1 and t}
$$
$$
O_t: \text{Total inventory level at end of period t}
$$
$$
N_t^{\leq 8000}: \text{Number of snitches (up to 8000) at the end of period t}
$$
$$
N_t^{\gt 8000}: \text{Number of snitches (over 8000)) at the end of period t}
$$

Cost:  
$$
C = \sum_{t = July}^{December}(1.50 \cdot I_t + 1.00 \cdot D_t + 0.10 \cdot S_t + 0.20 \cdot N_t^{\leq 8000} + 0.50 \cdot N_t^{\gt 8000})
$$

Objective:  
$$
\min{C}
$$

Subject To:  
$$
0 \leq N_t^{\leq 8000} \leq 8000
$$
$$
0 \leq N_t^{\gt 8000}
$$
$$
O_t = O_{t-1} + S_t - \text{demand}_t
$$
$$
O_t = N_t^{\leq 8000} + N_t^{\gt 8000}
$$
$$
S_t = S_{t-1} + I_t - D_t
$$
$$
S_t, I_t, D_t, O_t, N_t^{\leq 8000}, N_t^{\gt 8000} \geq 0
$$

##### Question 2-2

In [3]:
T = [:june, :july, :august, :september, :october, :november, :december]
demand = Dict(:july => 4000, :august => 8000, :september => 20000, :october => 12000, :november => 6000, :december => 2000)

model = Model(HiGHS.Optimizer)

@variable(model, S[T] >= 0)
@variable(model, I[T] >= 0)
@variable(model, D[T] >= 0)
@variable(model, O[T] >= 0)
@variable(model, NL[T] >= 0)
@variable(model, NG[T] >= 0)

@constraint(model, S[:june] == 4000)
@constraint(model, O[:june] == 2000)

@constraint(model, [t in T], NL[t] <= 8000)
@constraint(model, [t in T[2:end]], O[t] == O[T[findfirst(==(t), T)-1]] + S[t] - demand[t])
@constraint(model, [t in T], O[t] == NL[t] + NG[t])
@constraint(model, [t in T[2:end]], S[t] == S[T[findfirst(==(t), T)-1]] + I[t] - D[t])

@objective(model, Min, sum(1.50 * I[t] + 1.00 * D[t] + 0.10 * S[t] + 0.20 * NL[t] + 0.50 * NG[t] for t in T[2:end]))

set_silent(model)

optimize!(model)

println("Termination status: ", termination_status(model))
println("Cost: \$", objective_value(model))
println("Optimal production plan:")
for t in T[2:end]
    println("$t: Produce ", value(S[t]), " snitches, Inventory: ", value(O[t]))
end
for t in T[2:end]
    println("Month: ", t)
    println("  S (Production): ", value(S[t]))
    println("  I (Increase): ", value(I[t]))
    println("  D (Decrease): ", value(D[t]))
    println("  O (Inventory): ", value(O[t]))
    println("  NL (≤8000): ", value(NL[t]))
    println("  NG (>8000): ", value(NG[t]))
    println("------------------------")
end


Termination status: OPTIMAL
Cost: $25266.666666666664
Optimal production plan:
july: Produce 10000.0 snitches, Inventory: 8000.0
august: Produce 10666.666666666666 snitches, Inventory: 10666.666666666666
september: Produce 10666.666666666668 snitches, Inventory: 1333.3333333333321
october: Produce 10666.666666666668 snitches, Inventory: -0.0
november: Produce 8000.0 snitches, Inventory: 2000.0
december: Produce 8000.0 snitches, Inventory: 8000.0
Month: july
  S (Production): 10000.0
  I (Increase): 6000.0
  D (Decrease): 0.0
  O (Inventory): 8000.0
  NL (≤8000): 8000.0
  NG (>8000): 0.0
------------------------
Month: august
  S (Production): 10666.666666666666
  I (Increase): 666.6666666666661
  D (Decrease): 0.0
  O (Inventory): 10666.666666666666
  NL (≤8000): 8000.0
  NG (>8000): 2666.666666666666
------------------------
Month: september
  S (Production): 10666.666666666668
  I (Increase): 0.0
  D (Decrease): 0.0
  O (Inventory): 1333.3333333333321
  NL (≤8000): 1333.3333333333321

##### Question 2-3

Decision Variables:
$$
S_t: \text{number of snitches to produce during period t}
$$
$$
I_t: \text{The increase in production rates between periods t − 1 and t}
$$
$$
D_t: \text{The decrease in production rates between periods t − 1 and t}
$$
$$
O_t: \text{Total inventory level at end of period t}
$$
$$
N_t^{\leq 8000}: \text{Number of snitches (up to 8000) at the end of period t}
$$
$$
N_t^{\gt 8000}: \text{Number of snitches (over 8000)) at the end of period t}
$$
$$
B_t: \text{Total amount backlogged at end of period t t}
$$

Cost:  
$$
C = \sum_{t = July}^{December}(1.50 \cdot I_t + 1.00 \cdot D_t + 0.10 \cdot S_t + 0.20 \cdot N_t^{\leq 8000} + 0.50 \cdot N_t^{\gt 8000} + 0.25 \cdot B_t)
$$

Objective:  
$$
\min{C}
$$

Subject To:  
$$
0 \leq N_t^{\leq 8000} \leq 8000
$$
$$
0 \leq N_t^{\gt 8000}
$$
$$
O_t = O_{t-1} + S_t - \text{demand}_t + B_{t-1} - B_t
$$
$$
O_t = N_t^{\leq 8000} + N_t^{\gt 8000}
$$
$$
S_t = S_{t-1} + I_t - D_t
$$
$$
   \sum_{t = \text{July}}^{\text{December}} S_t = \sum_{t = \text{July}}^{\text{December}} \text{demand}_t
$$
$$
B_{December} = 0
$$
$$
S_t, I_t, D_t, O_t, N_t^{\leq 8000}, N_t^{\gt 8000}, B_t \geq 0
$$

In [4]:
T = [:june, :july, :august, :september, :october, :november, :december]
demand = Dict(:july => 4000, :august => 8000, :september => 20000, :october => 12000, :november => 6000, :december => 2000)

model = Model(HiGHS.Optimizer)

@variable(model, S[T] >= 0)
@variable(model, I[T] >= 0)
@variable(model, D[T] >= 0)
@variable(model, O[T] >= 0)
@variable(model, NL[T] >= 0)
@variable(model, NG[T] >= 0)
@variable(model, B[T] >= 0)


@constraint(model, S[:june] == 4000)
@constraint(model, O[:june] == 2000)
@constraint(model, B[:december] == 0)


@constraint(model, [t in T], NL[t] <= 8000)
@constraint(model, [t in T[2:end]], O[t] == O[T[findfirst(==(t), T)-1]] + S[t] - demand[t] - B[T[findfirst(==(t), T)-1]] + B[t])
@constraint(model, [t in T], O[t] == NL[t] + NG[t])
@constraint(model, [t in T[2:end]], S[t] == S[T[findfirst(==(t), T)-1]] + I[t] - D[t])
@constraint(model, sum(S[t] for t in T[2:end]) == sum(demand[t] for t in keys(demand)))

@objective(model, Min, sum(1.50 * I[t] + 1.00 * D[t] + 0.10 * S[t] + 0.20 * NL[t] + 0.50 * NG[t] + 0.25 * B[t] for t in T[2:end]))

set_silent(model)

optimize!(model)

println("Termination status: ", termination_status(model))
println("Cost: \$", objective_value(model))
println("Optimal production plan:")
for t in T[2:end]
    println("$t: Produce ", value(S[t]), " snitches, Inventory: ", value(O[t]))
end
for t in T[2:end]
    println("Month: ", t)
    println("  S (Production): ", value(S[t]))
    println("  I (Increase): ", value(I[t]))
    println("  D (Decrease): ", value(D[t]))
    println("  O (Inventory): ", value(O[t]))
    println("  NL (≤8000): ", value(NL[t]))
    println("  NG (>8000): ", value(NG[t]))
    println("------------------------")
end


Termination status: OPTIMAL
Cost: $19400.000000000004
Optimal production plan:
july: Produce 8666.666666666666 snitches, Inventory: 6666.666666666666
august: Produce 8666.666666666666 snitches, Inventory: 7333.333333333332
september: Produce 8666.666666666666 snitches, Inventory: -0.0
october: Produce 8666.666666666666 snitches, Inventory: -0.0
november: Produce 8666.666666666666 snitches, Inventory: -0.0
december: Produce 8666.666666666666 snitches, Inventory: 1999.9999999999964
Month: july
  S (Production): 8666.666666666666
  I (Increase): 4666.666666666666
  D (Decrease): 0.0
  O (Inventory): 6666.666666666666
  NL (≤8000): 6666.666666666666
  NG (>8000): 0.0
------------------------
Month: august
  S (Production): 8666.666666666666
  I (Increase): 0.0
  D (Decrease): 0.0
  O (Inventory): 7333.333333333332
  NL (≤8000): 7333.333333333332
  NG (>8000): 0.0
------------------------
Month: september
  S (Production): 8666.666666666666
  I (Increase): 0.0
  D (Decrease): 0.0
  O (Inven

##### Question 3-1

Decision Variables:
$$
x_t \quad \forall t \in \{1, 2, 3, 4, 5, 6\}
$$

Constraints: 
$$
3x_1 + 3x_2 + 4x_3 + 6x_6 = 15
$$
$$
2x_1 - 6x_2 - x_3 - 5x_4 + 2x_5 - x_6 = 10
$$
$$
x_1 - x_2 + x_3 - 3x_5 + x_6 = 2 
$$
$$
2x_1 + x_3 - x_4 + 6x_5 = 0
$$

Objective:
$$
\min{\sum_{i=1}^6|x_t|}
$$

##### Question 3-2

In [5]:
model = Model(HiGHS.Optimizer)

@variable(model, x[1:6]) 
@variable(model, y[1:6] >= 0) 

@constraint(model, y[1:6] >= x[1:6])
@constraint(model, y[1:6] >= -x[1:6])

@constraint(model, 3*x[1] + 3*x[2] + 4*x[3] + 6*x[6] == 15)
@constraint(model, 2*x[1] - 6*x[2] - x[3] - 5*x[4] + 2*x[5] - x[6] == 10)
@constraint(model, x[1] - x[2] + x[3] - 3*x[5] + x[6] == 2)
@constraint(model, 2*x[1] + x[3] - x[4] + 6*x[5] == 0)

@objective(model, Min, sum(y[t] for t in 1:6))

set_silent(model)
optimize!(model)

println("Termination status: ", termination_status(model))
println("Optimal solution found:")
for t in 1:6
    println("x_$t = ", value(x[t]))
    # println("y_$t = ", value(y[t]))
end

Termination status: OPTIMAL
Optimal solution found:
x_1 = -1.4109589041095891
x_2 = -0.0
x_3 = -0.0
x_4 = -3.232876712328767
x_5 = -0.06849315068493148
x_6 = 3.2054794520547945


##### Question 3-3

Decision Variables:
$$
x_t \quad \forall t \in \{1, 2, 3, 4, 5, 6\}
$$

Constraints: 
$$
3x_1 + 3x_2 + 4x_3 + 6x_6 = 15
$$
$$
2x_1 - 6x_2 - x_3 - 5x_4 + 2x_5 - x_6 = 10
$$
$$
x_1 - x_2 + x_3 - 3x_5 + x_6 = 2 
$$
$$
2x_1 + x_3 - x_4 + 6x_5 = 0
$$

Objective:
$$
\min{\max{|x_t|}}
$$

##### Question 3-4

In [6]:
model = Model(HiGHS.Optimizer)

@variable(model, x[1:6]) 
@variable(model, y[1:6] >= 0) 

@constraint(model, y[1:6] >= x[1:6])
@constraint(model, y[1:6] >= -x[1:6])

@constraint(model, 3*x[1] + 3*x[2] + 4*x[3] + 6*x[6] == 15)
@constraint(model, 2*x[1] - 6*x[2] - x[3] - 5*x[4] + 2*x[5] - x[6] == 10)
@constraint(model, x[1] - x[2] + x[3] - 3*x[5] + x[6] == 2)
@constraint(model, 2*x[1] + x[3] - x[4] + 6*x[5] == 0)

@objective(model, Min, y)

set_silent(model)
optimize!(model)

println("Termination status: ", termination_status(model))
println("Optimal solution found:")
for t in 1:6
    println("x_$t = ", value(x[t]))
    # println("y_$t = ", value(y[t]))
end

Termination status: DUAL_INFEASIBLE
Optimal solution found:
x_1 = 0.0
x_2 = 0.42696629213483145
x_3 = 0.4297752808988764
x_4 = -0.5646067415730338
x_5 = -0.1657303370786517
x_6 = -0.5


##### Question 3-5

Decision Variables:
$$
x_j \quad \forall j \in \{1, 2, 3, \dots, n\}
$$
$$
y_j \quad \forall j \in \{1, 2, 3, \dots, n\}
$$
$$
z_j \quad \forall j \in \{1, 2, 3, \dots, n\}
$$
Subject To:  
$$
x_j = y_j - z_j \quad \text{with} \quad y_j, z_j \geq 0
$$
$$
\sum_{j \in N} a_{ij} (y_j - z_j) = b_i \quad \forall i \in M
$$
$$
y_j, z_j \geq 0 \quad \forall j \in N
$$
Objective:
$$
\min{\sum_{j \in N} (y_j + z_j)}
$$

##### Question 4-1

Decision Variables:
$$
t_t \quad \text{ Start time of task t}\in T
$$
$$
d_t \quad \text{ Duration of task t}\in T
$$
$$
P \quad \text{\{(i, j) : i ∈ T directly precedes j ∈ T \}}
$$

Constraints:  
$$
t_j \geq t_i + d_i\quad \forall j \in  P_i \quad \forall i \in T
$$

Objective:  
$$
\min_{t} t_{18} + d_{18}
$$

##### Question 4-2

In [7]:
tasks = 1:18

dur = [2, 16, 9, 8, 10, 6, 2, 2, 9, 5, 3, 2, 1, 7, 4, 3, 9, 1]
duration = Dict(zip(tasks,dur))

pre = ( [], [1], [2], [2], [3], [4,5], [4], [6], [4,6], [4], [6], [9], [7], [2], [4,14], [8,11,14], [12], [17] )
pred = Dict(zip(tasks,pre))

model = Model(HiGHS.Optimizer)

@variable(model, tstart[tasks])

@constraint(model, [i in tasks, j in pred[i]], tstart[i] >= tstart[j] + duration[j])
@constraint(model, tstart[1] == 0)
@objective(model, Min, tstart[18] + duration[18])     

set_silent(model)
optimize!(model)
println("Termination Status: ", termination_status(model))
println(value.(tstart))
println("minimum duration: ", objective_value(model))

Termination Status: OPTIMAL
1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, 1:18
And data, a 18-element Vector{Float64}:
 -0.0
  2.0
 18.0
 18.0
 27.0
 37.0
 26.0
 43.0
 43.0
 26.0
 43.0
 52.0
 28.0
 18.0
 26.0
 46.0
 54.0
 63.0
minimum duration: 64.0


##### Question 4-3

Decision Variables:
$$
t_t \quad \text{ Start time of task t}\in T
$$
$$
d_t \quad \text{ Duration of task t}\in T
$$
$$
P \quad \text{\{(i, j) : i ∈ T directly precedes j ∈ T \}}
$$
$$
r_t \quad \text{ Weeks task t is completed early }\in T
$$
$$
m_t \quad \text{ Max \# of weeks task t can be reduced }
$$
$$
c_t \quad \text{ Cost per week to reduce task t }
$$
$$
\chi = 64 \quad \text{ Baseline time to complete the project}
$$

Constraints:  
$$
t_j \geq t_i + d_i - r_i\quad \forall j \in  P_i \quad \forall i \in T
$$
$$
0 \leq r_i \leq m_t \quad \forall j \in  P_i \quad \forall i \in T
$$
$$

$$

Objective:  
$$
\max 30(\chi - (x_{18} + d_{18})) - \sum_{i \in T}(r_i \cdot m_i)
$$

##### Question 4-4

In [8]:

tasks = 1:18

dur = [2, 16, 9, 8, 10, 6, 2, 2, 9, 5, 3, 2, 1, 7, 4, 3, 9, 1]
duration = Dict(zip(tasks,dur))

pre = ( [], [1], [2], [2], [3], [4,5], [4], [6], [4,6], [4], [6], [9], [7], [2], [4,14], [8,11,14], [12], [17] )
pred = Dict(zip(tasks,pre))

max_reduction = [0, 3, 1, 2, 2, 1, 1, 0, 2, 1, 1, 0, 0, 2, 2, 1, 3, 0]
cost_per_week = [0, 30, 26, 12, 17, 15, 8, 0, 42, 21, 18, 0, 0, 22, 12, 6, 16, 0]

m_reduction = Dict(zip(tasks, max_reduction))
c_cost = Dict(zip(tasks, cost_per_week))

model = Model(HiGHS.Optimizer)

@variable(model, tstart[tasks] >= 0 )
@variable(model,r[tasks] >= 0) 

@constraint(model,[t in tasks], r[t] <= m_reduction[t])
@constraint(model, [i in tasks, j in pred[i]], tstart[i] >= tstart[j] + duration[j] - r[j])
@constraint(model, tstart[1] == 0)
@objective(model, Max, 30*(64 - (tstart[18] + duration[18])) - sum(c_cost[i] * r[i] for i in tasks))

set_silent(model)
optimize!(model)
println(value.(tstart))
println("Maximun Profit: ", objective_value(model))

1-dimensional DenseAxisArray{Float64,1,...} with index sets:
    Dimension 1, 1:18
And data, a 18-element Vector{Float64}:
  0.0
  2.0
 18.0
 18.0
 26.0
 34.0
 26.0
 39.0
 39.0
 26.0
 39.0
 48.0
 28.0
 18.0
 26.0
 42.0
 50.0
 56.0
Maximun Profit: 87.0


##### Question 5-1

Decision Variables:
$$
R: \text{ Set of restaurants }
$$
$$
d_{ij}: \text{ Distance from restaurant i to restaurant j}
$$
$$
P: \text{ Path taken (start, end) }
$$
$$
x_{ij} \text{ 1 if the path from i to j is taken otherwise 0}
$$

Constraints:  
$$
\sum_{j} x_{\text{ss},j} = 1
$$
$$
\sum_{i} x_{i, \text{a}} = 1
$$
$$
\sum_{i} x_{ik} = \sum_{j} x_{kj} \quad \forall k \neq \text{Short Stack}, \text{Alchemy}
$$

Objective:  
$$
\min{\sum_{(i,j) \in P}d_{ij} \cdot x_{ij}}
$$

##### Question 5-2

In [9]:
restaurants_end = [ :gd,  :of,  :rs, :iag, :wt, :a]
restaurants_start = [:ss, :gd, :of, :rs, :iag, :wt]

# restaurants_y -> restaurants_x == OKAY
# restaurants_x -> restaurants_y == BAD
dist = NamedArray(
    [4    6   5   100000   100000   100000; 
     100000  1   100000 7     100000   100000; 
     100000  100000 2   5.5   4     100000; 
     100000  100000 100000 100000   5     100000; 
     100000  100000 100000 100000   1     6; 
     100000  100000 100000 100000   100000   8],
    (restaurants_start, restaurants_end),
    ("restaurant", "restaurant")
)

model = Model(HiGHS.Optimizer)

@variable(model, x[restaurants_start, restaurants_end], Bin) # is the drive taken start -> end

@objective(model, Min, sum(dist[i, j] * x[i, j] for i in restaurants_start for j in restaurants_end))

@constraint(model, sum(x[:ss, j] for j in restaurants_end) == 1)
@constraint(model, sum(x[i, :a] for i in restaurants_start) == 1)
for k in setdiff(restaurants_start, [:ss, :a])
    @constraint(model, sum(x[i, k] for i in restaurants_start) == sum(x[k, j] for j in restaurants_end))
end

set_silent(model)
optimize!(model)

if termination_status(model) == MOI.OPTIMAL
    println("Optimal Solution Found!")
    println("Shortest Path length: ", objective_value(model))
    for i in restaurants_start
        for j in restaurants_end
            if value(x[i, j]) > 0.5
                println("Path from $i to $j")
            end
        end
    end
else
    println("No optimal solution found.")
end

Optimal Solution Found!
Shortest Path length: 16.5
Path from ss to gd
Path from gd to of
Path from of to iag
Path from iag to a
